# 03. Statistical Analysis & Outlier Detection
This notebook focuses on advanced statistical calculations for task productivity.
It covers summary statistics (mean, median, std, percentiles), Pearson correlation analysis, and Interquartile Range (IQR) outlier detection.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add backend to path
sys.path.append(os.path.abspath('../backend'))
from app.analytics.queries import get_tasks_analytics_dataframe
from app.analytics.cleaning import clean_tasks_dataframe
from app.analytics.features import enrich_task_features
from app.analytics.statistics import (
    calculate_descriptive_stats,
    calculate_correlations,
    detect_outliers_iqr
)
from app.db.session import engine
from sqlalchemy.orm import sessionmaker

SessionLocal = sessionmaker(bind=engine)
db = SessionLocal()
df_raw = get_tasks_analytics_dataframe(db)
df_clean, _ = clean_tasks_dataframe(df_raw)
df = enrich_task_features(df_clean)
print(f'Enriched dataset loaded: {len(df)} rows.')

## 1. Descriptive Statistics (Percentiles P25, P50, P75, P90)

In [ ]:
est_stats = calculate_descriptive_stats(df['estimated_hours'])
act_stats = calculate_descriptive_stats(df['actual_hours'])
comp_days_stats = calculate_descriptive_stats(df['completion_days'])
err_stats = calculate_descriptive_stats(df['estimation_error'])

stats_summary = pd.DataFrame({
    'Estimated Hours': est_stats,
    'Actual Hours': act_stats,
    'Completion Days': comp_days_stats,
    'Estimation Error': err_stats
})
print(stats_summary)

## 2. Correlation Matrix (Pearson Correlations)

In [ ]:
corrs = calculate_correlations(df)
print('Pearson Correlation Coefficients:')
for k, v in corrs.items():
    print(f'  {k}: {v}')

## 3. IQR Outlier Detection (Interquartile Range)

In [ ]:
est_outliers = detect_outliers_iqr(df['estimated_hours'])
act_outliers = detect_outliers_iqr(df['actual_hours'])

print('Estimated Hours Outlier Analysis:')
print(f"  Q1={est_outliers['q1']}, Q3={est_outliers['q3']}, IQR={est_outliers['iqr']}")
print(f"  Bounds: [{est_outliers['lower_bound']}, {est_outliers['upper_bound']}]")
print(f"  Outliers Count: {est_outliers['outlier_count']}")

print('\nActual Hours Outlier Analysis:')
print(f"  Q1={act_outliers['q1']}, Q3={act_outliers['q3']}, IQR={act_outliers['iqr']}")
print(f"  Bounds: [{act_outliers['lower_bound']}, {act_outliers['upper_bound']}]")
print(f"  Outliers Count: {act_outliers['outlier_count']}")